# 03 · OCR — Step 15
Scanned pages (no text layer) → OCR → quality gate → page records → letters.

| Cell | What it does |
|---|---|
| 15.0 | Setup: imports, settings, load `pages_tables.jsonl` |
| 15.1 | Functions: render page → OCR lines → drop junk lines → paragraphs |
| 15.2 | Look at ONE page (learning cell) |
| 15.3 | OCR all candidate pages once → `ocr_lines.jsonl` (fingerprint cache: re-runs OCR only if PDF/settings change) |
| 15.4 | Quality gate: thresholds, manual overrides, status per page |
| 15.5 | Threshold sweep (explore only, changes nothing) |
| 15.6 | Build OCR page records (text or "please view page N" notice) |
| 15.7 | Save images of rejected pages to check by eye |
| 15.8 | Letters from the TOC: letterhead / body / sign-off rules (registries) |
| 15.9 | Small text fixes on letter bodies (ordinals, `\|`→I, bullets) → build `LETTERS` |
| 15.T | Tests (step 15 OCR + 15.8 letters + 15.9 fixes + 15.H pages) |
| 15.H | Save `pages_ocr.jsonl` + `letters.jsonl` — only when all tests pass, then reload check |

## 15.0 Setup

In [1]:
import os, re, json, copy, time, hashlib
from collections import Counter
import pandas as pd
import pdfplumber
import pytesseract
from src.paths import ROOT, PDF_PATH, INTERIM, PAGES_TABLES, PAGES_OCR, OCR_LINES, TOC_CSV, LETTERS_JSONL
from src.text_utils import normalize_chars

pd.set_option("display.max_colwidth", 100)

# Tesseract's location depends on the machine → environment variable, with the usual Windows path as default
TESSERACT_CMD = os.environ.get("TESSERACT_CMD", r"C:\Program Files\Tesseract-OCR\tesseract.exe")
pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD
print("Tesseract version:", pytesseract.get_tesseract_version())

OCR_DPI  = 300        # image sharpness for OCR
OCR_LANG = "eng"

PAGES          = [json.loads(line) for line in PAGES_TABLES.open(encoding="utf-8")]
PAGE_BY_NO     = {p["pdf_page"]: p for p in PAGES}
PRINTED        = {p["pdf_page"]: p["printed_page"] for p in PAGES}
OCR_CANDIDATES = [p["pdf_page"] for p in PAGES if p["method"] == "needs_ocr"]     # from the data, not typed
print(f"{len(PAGES)} pages loaded · {len(OCR_CANDIDATES)} need OCR")

CHECK_DIR = INTERIM / "ocr_check"          # images saved here for checking by eye
CHECK_DIR.mkdir(exist_ok=True)

Tesseract version: 5.5.3.20260724
174 pages loaded · 56 need OCR


## 15.1 Functions: page → lines → drop junk → paragraphs
- `ocr_lines`: one row per line, with confidence. Words below `WORD_MIN_CONF` are removed from the line.
- Line drop rules (registry, first match wins): **no real word** (side-label letters, symbols) · **low confidence** (Hindi letterhead read as English)
- `to_paragraphs`: joins kept lines using Tesseract's own paragraph ids, re-joins `recruit-` + `ment`

In [2]:
WORD_MIN_CONF = 40      # a word below this is dropped from its line
LINE_MIN_CONF = 60      # a line whose average is below this is dropped
REAL_WORD     = re.compile(r"[A-Za-z]{3,}")

def render_page(pdf_no, dpi=OCR_DPI):
    """PDF page → picture (PIL image). OCR can only read pictures."""
    with pdfplumber.open(PDF_PATH) as pdf:
        return pdf.pages[pdf_no - 1].to_image(resolution=dpi).original

def ocr_lines(img, lang=OCR_LANG):
    """OCR → one row per line: raw (all words), text (confident words only), line confidence."""
    d = pytesseract.image_to_data(img, lang=lang, output_type=pytesseract.Output.DATAFRAME)
    d["text"] = d["text"].fillna("").astype(str).str.strip()
    d = d[(d["conf"] >= 0) & (d["text"] != "")].copy()
    d["kept"] = d["text"].where(d["conf"] >= WORD_MIN_CONF, "")
    lines = (d.groupby(["block_num", "par_num", "line_num"], sort=True)
               .agg(conf=("conf", "mean"), top=("top", "min"), n_words=("text", "size"),
                    raw=("text", " ".join),
                    text=("kept", lambda s: " ".join(t for t in s if t)))
               .reset_index())
    lines["conf"] = lines["conf"].round(1)
    return lines

# ---- line drop rules (registry: first rule that says "drop" wins) ----
LINE_DROP_RULES = {}
def drop_rule(name):
    def deco(fn):
        LINE_DROP_RULES[name] = fn
        return fn
    return deco

@drop_rule("no real word")
def no_real_word(line):
    return not REAL_WORD.search(line["text"])

@drop_rule("low confidence")
def low_confidence(line):
    return line["conf"] < LINE_MIN_CONF

def mark_dropped(lines):
    lines = lines.copy()
    lines["dropped_by"] = [next((name for name, rule in LINE_DROP_RULES.items() if rule(row)), "")
                           for _, row in lines.iterrows()]
    return lines

def to_paragraphs(lines):
    """Kept lines → paragraphs (Tesseract's paragraph ids); re-joins words split with a hyphen."""
    kept, paras = lines[lines["dropped_by"] == ""], []
    for _, g in kept.groupby(["block_num", "par_num"], sort=True):
        text = ""
        for t in g["text"]:
            text = text[:-1] + t if re.search(r"[a-z]-$", text) else (text + " " + t).strip()
        paras.append({"text": text, "conf": round(float(g["conf"].mean()), 1), "top": int(g["top"].min())})
    return paras

def first_drop(line):
    """Name of the first rule that drops this line, or '' if it is kept (used by tests)."""
    return next((name for name, rule in LINE_DROP_RULES.items() if rule(line)), "")


## 15.2 Look at ONE page (learning cell — change `PDF_NO` to explore)

In [3]:
PDF_NO = 112

lines = mark_dropped(ocr_lines(render_page(PDF_NO)))
print("DROPPED:")
print(lines[lines["dropped_by"] != ""][["conf", "dropped_by", "raw"]].to_string())
print("\nKEPT → PARAGRAPHS:")
for p in to_paragraphs(lines):
    print(f"[{p['conf']}] {p['text'][:140]}")

DROPPED:
    conf      dropped_by                     raw
0   68.7    no real word                     WY)
1   49.5    no real word                       a
2   43.2    no real word                      Lu
3   24.5    no real word                      FE
4   21.0    no real word                       _
5   59.1    no real word                      Lu
6   65.6    no real word                      —!
7    3.8    no real word                     im”
8   82.4    no real word                       O
9   89.2    no real word                       =
10  72.9    no real word                       2
11  63.5    no real word                      a)
12  61.7    no real word                      Zz
13  45.0    no real word                      im
14  47.1    no real word                       a
15  87.9    no real word                       =
16  83.8    no real word                       O
17  90.4    no real word                       O
20  18.1    no real word              seeeel fon
21  27.9  l

## 15.3 OCR all candidate pages once, save, reuse (fingerprint cache)

In [4]:
OCR_FINGERPRINT = {   # if any of these change, the saved OCR is stale → run OCR again
    "pdf": hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()[:16],
    "dpi": OCR_DPI, "lang": OCR_LANG, "word_min_conf": WORD_MIN_CONF,
}
LINE_COLUMNS = ["block_num", "par_num", "line_num", "conf", "top", "n_words", "raw", "text"]

def lines_frame(rows):
    """Rows → DataFrame with fixed columns (a blank page gives [] → still has all columns)."""
    return pd.DataFrame(rows, columns=LINE_COLUMNS)

def load_or_run_ocr(pages):
    if OCR_LINES.exists():
        with OCR_LINES.open(encoding="utf-8") as f:
            header, *saved = [json.loads(l) for l in f]
        if header.get("fingerprint") == OCR_FINGERPRINT and sorted(r["pdf_page"] for r in saved) == sorted(pages):
            print(f"loaded saved OCR for {len(saved)} pages")
            return {r["pdf_page"]: mark_dropped(lines_frame(r["lines"])) for r in saved}
        print("saved OCR is stale → running OCR again")

    raw, start = {}, time.time()
    for i, n in enumerate(pages, 1):
        raw[n] = ocr_lines(render_page(n))[LINE_COLUMNS]
        print(f"\r{i}/{len(pages)} pages · {time.time() - start:.0f}s", end="")
    with OCR_LINES.open("w", encoding="utf-8") as f:
        f.write(json.dumps({"fingerprint": OCR_FINGERPRINT}) + "\n")
        for n, df in raw.items():
            f.write(json.dumps({"pdf_page": n, "lines": json.loads(df.to_json(orient="records"))}) + "\n")
    print(f"\nsaved → {OCR_LINES.relative_to(ROOT)}")
    return {n: mark_dropped(df) for n, df in raw.items()}

OCR_RESULTS = load_or_run_ocr(OCR_CANDIDATES)

loaded saved OCR for 56 pages


## 15.4 Quality gate
- `OCR_ACCEPT`: all thresholds in one place
- `OCR_OVERRIDES`: pages decided by eye (later → `config/manual_decisions.json`)
- Checks (registry, all must pass) → status (registry, first match wins): `blank` · `low_quality` · `ok`

In [5]:
OCR_ACCEPT = {
    "blank_words":    10,     # fewer readable words → blank page
    "min_words":      30,     # enough words to be useful
    "min_conf":       80.0,   # average confidence of the kept text
    "max_junk_share": 0.5,    # share of all words thrown away
}
OCR_OVERRIDES = {             # rejected after checking by eye
    127: "low_quality",       # letter continues on 128
    128: "low_quality",
}
LOW_QUALITY_MSG = ("This scanned page is too unclear to read reliably. "
                   "Please view PDF page {pdf_page} (printed page {printed}).")

ACCEPT_CHECKS = {}
def accept_check(name):
    def deco(fn):
        ACCEPT_CHECKS[name] = fn
        return fn
    return deco

@accept_check("enough words")
def enough_words(q):    return q["kept_words"] >= OCR_ACCEPT["min_words"]

@accept_check("confident text")
def confident_text(q):  return q["kept_conf"] >= OCR_ACCEPT["min_conf"]

@accept_check("not mostly junk")
def not_mostly_junk(q): return q["junk_share"] <= OCR_ACCEPT["max_junk_share"]

STATUS_RULES = {
    "blank":       lambda q: q["kept_words"] < OCR_ACCEPT["blank_words"],
    "low_quality": lambda q: bool(q["failed_checks"]),
    "ok":          lambda q: True,
}

def page_quality(n, lines):
    kept       = lines[lines["dropped_by"] == ""]
    all_words  = int(lines["n_words"].sum())
    kept_words = int(kept["text"].str.split().str.len().sum())
    q = {"pdf_page": n, "printed": PRINTED.get(n),
         "words": all_words, "kept_words": kept_words,
         "kept_conf":  round(float(kept["conf"].mean()), 1) if len(kept) else 0.0,
         "junk_share": round(1 - kept_words / all_words, 2) if all_words else 1.0}
    q["failed_checks"] = [name for name, test in ACCEPT_CHECKS.items() if not test(q)]
    q["failed_checks"] += ["manual review"] * (n in OCR_OVERRIDES)
    q["status"] = OCR_OVERRIDES.get(n, next(s for s, rule in STATUS_RULES.items() if rule(q)))
    return q

QUALITY = pd.DataFrame([page_quality(n, df) for n, df in OCR_RESULTS.items()]).sort_values("kept_conf")
QUALITY.to_csv(ROOT / "quality" / "ocr_quality.csv", index=False)
Q_BY_PAGE = {q["pdf_page"]: q for q in QUALITY.to_dict("records")}

print(QUALITY["status"].value_counts().to_string(), "\n")
QUALITY[QUALITY["status"] != "ok"]

status
ok             39
blank          12
low_quality     5 



,pdf_page,printed,words,kept_words,kept_conf,junk_share,failed_checks,status
1,2,NaN,0,0,0.0,1.00,"[enough words, confident text, not mostly junk]",blank
3,4,ii,0,0,0.0,1.00,"[enough words, confident text, not mostly junk]",blank
7,8,vi,0,0,0.0,1.00,"[enough words, confident text, not mostly junk]",blank
5,6,iv,0,0,0.0,1.00,"[enough words, confident text, not mostly junk]",blank
11,12,x,0,0,0.0,1.00,"[enough words, confident text, not mostly junk]",blank
13,14,xii,0,0,0.0,1.00,"[enough words, confident text, not mostly junk]",blank
9,10,viii,0,0,0.0,1.00,"[enough words, confident text, not mostly junk]",blank
55,174,154,0,0,0.0,1.00,"[enough words, confident text, not mostly junk]",blank
54,173,153,0,0,0.0,1.00,"[enough words, confident text, not mostly junk]",blank
33,128,108,134,61,73.5,0.54,"[confident text, not mostly junk, manual review]",low_quality


## 15.5 Threshold sweep (explore only — changes nothing)

In [6]:
base = (QUALITY["kept_words"] >= OCR_ACCEPT["min_words"]) & (QUALITY["junk_share"] <= OCR_ACCEPT["max_junk_share"])
print("min_conf → pages accepted")
for c in (70, 75, 80, 85, 90):
    print(f"   {c}: {int((base & (QUALITY['kept_conf'] >= c)).sum())}")

min_conf → pages accepted
   70: 40
   75: 40
   80: 40
   85: 39
   90: 35


## 15.6 Build OCR page records (text for `ok`, notice for `low_quality`)

In [7]:
STATUS_CONTENT = {        # status → what goes into the page's lines
    "ok":          lambda lines: [{"text": normalize_chars(p["text"]), "level": "body", "font": None,
                                   "size": None, "top": p["top"], "conf": p["conf"]}
                                  for p in to_paragraphs(lines)],
    "low_quality": lambda lines: [],
    "blank":       lambda lines: [],
}
STATUS_NOTICE = {         # status → message for the chatbot
    "ok":          lambda q: None,
    "low_quality": lambda q: LOW_QUALITY_MSG.format(pdf_page=q["pdf_page"], printed=q["printed"]),
    "blank":       lambda q: None,
}

def ocr_page_record(page, lines, q):
    rec = copy.deepcopy(page)
    rec["method"]      = "ocr"
    rec["status"]      = q["status"]
    rec["ocr_quality"] = {k: q[k] for k in ("words", "kept_words", "kept_conf", "junk_share", "failed_checks")}
    rec["lines"]       = STATUS_CONTENT[q["status"]](lines)
    rec["notice"]      = STATUS_NOTICE[q["status"]](q)
    return rec

OCR_PAGES = {n: ocr_page_record(PAGE_BY_NO[n], OCR_RESULTS[n], Q_BY_PAGE[n]) for n in OCR_RESULTS}

def show_ocr(n, k=4):
    r = OCR_PAGES[n]
    print(f"\nPDF {n} (printed {r['printed_page']}) · {r['status']} · {r['ocr_quality']}")
    print("   notice:", r["notice"])
    for l in r["lines"][:k]:
        print("   ", l["text"][:110])

for n in [112, 127, 128]:
    show_ocr(n)


PDF 112 (printed 92) · ok · {'words': 263, 'kept_words': 225, 'kept_conf': 90.3, 'junk_share': 0.14, 'failed_checks': []}
   notice: None
    4. Dated: 3° February, 2016,Strengthening Specialist Support in Public Health facilities
    waa, ag feeeil - 110011 GOVERNMENT OF INDIA
    aa MINISTRY OF HEALTH & FAMILY WELFARE NIRMAN BHAVAN, NEW DELHI - 110011
    C.K. Mishra Additional Secretary &

PDF 127 (printed 107) · low_quality · {'words': 600, 'kept_words': 454, 'kept_conf': 82.6, 'junk_share': 0.24, 'failed_checks': ['manual review']}
   notice: This scanned page is too unclear to read reliably. Please view PDF page 127 (printed page 107).

PDF 128 (printed 108) · low_quality · {'words': 134, 'kept_words': 61, 'kept_conf': 73.5, 'junk_share': 0.54, 'failed_checks': ['confident text', 'not mostly junk', 'manual review']}
   notice: This scanned page is too unclear to read reliably. Please view PDF page 128 (printed page 108).


## 15.7 Save images of rejected pages → check by eye

In [8]:
rejected = QUALITY.loc[QUALITY["status"] == "low_quality", "pdf_page"].tolist()
for n in rejected:
    img = render_page(n)
    img.resize((img.width // 2, img.height // 2)).save(CHECK_DIR / f"page_{n}.png")
print(f"{len(rejected)} rejected pages saved → {CHECK_DIR.relative_to(ROOT)}: {rejected}")

5 rejected pages saved → data\interim\ocr_check: [128, 132, 127, 1, 3]


## 15.8 Letters from the TOC
TOC gives number / date / subject / first page. Each OCR line gets a label (registries, first match wins): **letterhead** (dropped anywhere: caption, D.O. No, Telefax/E-mail, Nirman Bhavan; before the body: anything that is not a sentence), **sign-off** (Yours sincerely, short `(Name)` line, Addressed/Copy to → rest of that page dropped), else **body**. Enclosure pages after the sign-off page are kept. A letter is `ok` only if ALL its pages passed the quality gate.

In [9]:
# ---- which line is letterhead / sign-off? (registries, first match wins) ----
LETTERHEAD_ANYWHERE = {            # never real body text → dropped wherever they appear
    "caption":   r"^(\d+\.\s*)?Dated\b",          # "3. Dated: 21* April…" — number optional (OCR can drop it)
    "D.O. no":   r"^\W*D\.?\s*O\.?\s*No",
    "contact":   r"Telefax|\bFax\s*:|E-?mail\s*:",
    "address":   r"NIRMAN BHAVAN|\b1100[17]1\b",
    "page no":   r"^Page \d+ of \d+$",
}
LETTERHEAD_TOP = {                 # before the body starts: always letterhead
    "greeting":  r"^Dear\b",
    "subject":   r"^Subject\b",
}
SENTENCE_WORDS = {"the", "is", "are", "has", "have", "been", "to", "be", "in", "that",
                  "this", "you", "we", "will", "may", "would"}
SIGN_OFF = {                       # first match ends the letter part of that page
    "yours":     r"^\W*Yours\b|\bYours?\s+(sincerely|faithfully)",
    "name":      r"^.{0,40}\)\s*$",                # "(Manoj Jhalani)", "anoj Jhalani )" — short line ending ")"
    "addressed": r"^(Addressed to|Copy to)\b",
}
BODY_MIN_WORDS = 10

def looks_like_sentence(text):
    """Body starts at the first long line that reads like a sentence (letterhead lines are
    names/addresses: 'Government of India Ministry of Health & Family Welfare …')."""
    words = [w.strip(",.;:()").lower() for w in text.split()]
    return len(words) >= BODY_MIN_WORDS and len(SENTENCE_WORDS & set(words)) >= 2

def first_match(text, rules):
    return next((name for name, pat in rules.items() if re.search(pat, text, flags=re.I)), "")

def is_sign_off(text):
    name = first_match(text, SIGN_OFF)
    return name if (name != "name" or len(text.split()) <= 6) else ""

def label_letter_lines(pages):
    """PDF pages of one letter → [(pdf_page, text, label)]; label '' = body line."""
    out, body_started, signed = [], False, False
    for n in pages:
        page_signed = False
        for line in OCR_PAGES[n]["lines"]:
            t = line["text"]
            if page_signed:
                label = "after sign-off"
            elif first_match(t, LETTERHEAD_ANYWHERE):
                label = "letterhead: " + first_match(t, LETTERHEAD_ANYWHERE)
            elif not body_started and (first_match(t, LETTERHEAD_TOP) or not looks_like_sentence(t)):
                label = "letterhead: " + (first_match(t, LETTERHEAD_TOP) or "before body")
            elif body_started and not signed and is_sign_off(t):
                label, page_signed, signed = "sign-off: " + is_sign_off(t), True, True
            else:
                label, body_started = "", True
            out.append((n, t, label))
    return out

# ---- letters from the TOC ----
TOC     = pd.read_csv(TOC_CSV, dtype=str)
LET_TOC = TOC[TOC["kind"] == "letter"].copy()
LET_TOC["pdf_start"] = LET_TOC["pdf_start"].astype(int)
ANNEX_II = TOC[(TOC["kind"] == "annexure") & (TOC["number"] == "II")].iloc[0]
LAST_LETTER_PAGE = int(ANNEX_II["printed_end"]) + 20                  # printed = PDF − 20
TITLE_RE = re.compile(r"^Dated:\s*(?P<date>.+?\d{4}),\s*(?P<subject>.+)$")
UNREADABLE_MSG = ("This letter's scan is too unclear to read reliably. "
                  "Please view PDF pages {first}-{last} (printed pages {p_first}-{p_last}).")

def letter_pages(i):
    start = LET_TOC["pdf_start"].iloc[i]
    end = LET_TOC["pdf_start"].iloc[i + 1] - 1 if i + 1 < len(LET_TOC) else LAST_LETTER_PAGE
    return list(range(start, end + 1))

def build_letter(i):
    row, pages = LET_TOC.iloc[i], letter_pages(i)
    m = TITLE_RE.match(row["title"])
    page_status = {str(n): OCR_PAGES[n]["status"] for n in pages}
    ok = all(s == "ok" for s in page_status.values())
    body = "\n\n".join(t for _, t, label in label_letter_lines(pages) if label == "") if ok else ""
    return {
        "doc_type": "letter", "letter_no": int(row["number"]),
        "date_text": m["date"],
        "date": pd.to_datetime(re.sub(r"(\d)(st|nd|rd|th)", r"\1", m["date"])).strftime("%Y-%m-%d"),
        "subject": m["subject"],
        "pdf_pages": pages, "printed_pages": [PRINTED[n] for n in pages],
        "page_status": page_status, "status": "ok" if ok else "unreadable",
        "notice": None if ok else UNREADABLE_MSG.format(first=pages[0], last=pages[-1],
                                                        p_first=PRINTED[pages[0]], p_last=PRINTED[pages[-1]]),
        "body": fix_text(body),                                        # 15.9
        "search": ok,
    }

def show_letter(no):
    """Learning helper: every line of a letter with the label it got."""
    for n, t, label in label_letter_lines(letter_pages(no - 1)):
        print(f"p{n} {label or 'BODY':<22} | {t[:90]}")

show_letter(3)      # change the number to look at any letter

p111 letterhead: caption    | Dated: 21* April, 2015, JS(Policy) Gol or NHSRC Observers for recruitment process
p111 letterhead: address    | frat was, ag - 110011
p111 letterhead: address    | GOVERNMENT OF INDIA MINISTRY OF HEALTH & FAMILY WELFARE NIRMAN BHAVAN, NEW DELHI - 110011
p111 letterhead: before body | Manoj Jhalani, las
p111 letterhead: before body | Joint Secretary
p111 letterhead: contact    | Telefax : 23063687
p111 letterhead: contact    | E-mail : manoj.jhalani@nic.in
p111 letterhead: D.O. no    | D.O. No. 7(45)/2014-NRHM-I 21° Apyib 2015
p111 BODY                   | I would like to draw your attention to the D.O. letters of even number dated 22™ September
p111 BODY                   | In this regard, it is also reiterated that competency based assessment was also a key cond
p111 BODY                   | It is requested that an action taken report in respect of HR recruitment and competency ba
p111 sign-off: yours        | Yours sincerely,
p111 after sign-off         

## 15.9 Small text fixes (letter bodies) → build `LETTERS`
Registry `TEXT_FIXES`, applied per paragraph: OCR'd superscript ordinals (`21*`, `22™`, `19"`, `3°`, `22"4`) → `21st`…, `| hope` → `I hope`, bullets read as `e`/`.`/`*` → `- `.

In [10]:
MONTHS = r"(?:January|February|March|April|May|June|July|August|September|October|November|December)"

def ordinal(n):
    n = int(n)
    return f"{n}{'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')}"

# registry: name → (pattern, replacement); applied in order to every paragraph of a letter body
TEXT_FIXES = {
    # superscript "st/nd/rd/th" read as a symbol: 21* · 22™ · 19" · 3° · 22"4 · 13'" · 19%
    "ordinal":      (re.compile(rf"\b(\d{{1,2}})\s?(?:\"4|'\"|[\"'*°%™])(?=\s*{MONTHS}\b)"),
                     lambda m: ordinal(m[1])),
    "I read as |":  (re.compile(r"^\|\s"), "I "),
    "bullet":       (re.compile(r"^(?:e(?=\s+[A-Z])|[.*«®])\s+"), "- "),
    "stray mark":   (re.compile(r"^[»>]\s+"), ""),
}

def fix_text(text):
    paras = []
    for p in text.split("\n\n"):
        for pattern, repl in TEXT_FIXES.values():
            p = pattern.sub(repl, p)
        paras.append(p)
    return "\n\n".join(paras)

LETTERS = [build_letter(i) for i in range(len(LET_TOC))]
print(f"{len(LETTERS)} letters · {Counter(l['status'] for l in LETTERS)}")

21 letters · Counter({'ok': 19, 'unreadable': 2})


## 15.T Tests

In [11]:
TESTS = []
def check(group, name, get_actual, expected, desc=None):
    try:
        actual = get_actual()
        ok = expected(actual) if callable(expected) else actual == expected
    except Exception as e:
        actual, ok = f"ERROR {type(e).__name__}: {e}", False
    TESTS.append({"group": group, "test": name, "result": "PASS" if ok else "FAIL",
                  "expected": desc if callable(expected) else expected, "actual": actual})

G = "15 ocr"
check(G, "OCR candidates", lambda: len(OCR_CANDIDATES), 56)
check(G, "every candidate OCR'd", lambda: sorted(OCR_RESULTS) == sorted(OCR_CANDIDATES), True)
check(G, "statuses are known", lambda: sorted(set(QUALITY["status"]) - set(STATUS_RULES)), [])
check(G, "PDF 112 accepted", lambda: OCR_PAGES[112]["status"], "ok")
check(G, "PDF 127 + 128 rejected by manual review",
      lambda: [(n, OCR_PAGES[n]["status"], "manual review" in OCR_PAGES[n]["ocr_quality"]["failed_checks"]) for n in (127, 128)],
      [(127, "low_quality", True), (128, "low_quality", True)])
check(G, "every rejected page has a notice",
      lambda: [n for n, r in OCR_PAGES.items() if r["status"] == "low_quality" and not r["notice"]], [])
check(G, "every accepted page has text",
      lambda: [n for n, r in OCR_PAGES.items() if r["status"] == "ok" and not r["lines"]], [])
check(G, "no text on rejected or blank pages",
      lambda: [n for n, r in OCR_PAGES.items() if r["status"] != "ok" and r["lines"]], [])
check(G, "accepted pages meet min_conf (unless overridden)",
      lambda: [n for n, r in OCR_PAGES.items() if r["status"] == "ok"
               and r["ocr_quality"]["kept_conf"] < OCR_ACCEPT["min_conf"]], [])

G = "15 unit"
check(G, "drop rule: side-label letter", lambda: first_drop({"text": "O", "conf": 90.4}), "no real word")
check(G, "drop rule: Hindi read as English", lambda: first_drop({"text": "seeeel fon", "conf": 18.1}), "low confidence")
check(G, "drop rule: body line kept", lambda: first_drop({"text": "There is a significant shortage", "conf": 95.0}), "")
check(G, "paragraph joins hyphen split",
      lambda: to_paragraphs(pd.DataFrame([
          {"block_num": 1, "par_num": 1, "line_num": 1, "conf": 95.0, "top": 10, "text": "the recruit-", "dropped_by": ""},
          {"block_num": 1, "par_num": 1, "line_num": 2, "conf": 95.0, "top": 20, "text": "ment drive.", "dropped_by": ""}]))[0]["text"],
      "the recruitment drive.")

# ---------- 15.8 letters ----------
G = "15.8 letters"
BODY_OF = {l["letter_no"]: l["body"] for l in LETTERS}
check(G, "21 letters from the TOC", lambda: [l["letter_no"] for l in LETTERS], list(range(1, 22)))
check(G, "letter pages = PDF 109-145, each once",
      lambda: sorted(n for l in LETTERS for n in l["pdf_pages"]), list(range(109, 146)))
check(G, "letters 11 + 14 unreadable: no text, notice, not searchable",
      lambda: [(l["letter_no"], bool(l["body"]), bool(l["notice"]), l["search"]) for l in LETTERS if l["status"] != "ok"],
      [(11, False, True, False), (14, False, True, False)])
check(G, "every readable letter has a body",
      lambda: [l["letter_no"] for l in LETTERS if l["status"] == "ok" and not l["body"]], [])
check(G, "no letterhead line left in bodies",
      lambda: [(no, first_match(p, LETTERHEAD_ANYWHERE)) for no, b in BODY_OF.items()
               for p in b.split("\n\n") if first_match(p, LETTERHEAD_ANYWHERE)], [])
check(G, "no 'Yours sincerely' left in bodies",
      lambda: [no for no, b in BODY_OF.items() if re.search(SIGN_OFF["yours"], b, flags=re.I)], [])
check(G, "letter 3: body starts after letterhead (caption lost its number)",
      lambda: BODY_OF[3][:35], "I would like to draw your attention")
check(G, "letter 16: first body line kept", lambda: BODY_OF[16][:33], "As you are aware, Sub Health Cent")
check(G, "letter 4: enclosure pages kept",
      lambda: "Mechanisms for Strengthening Specialist Support" in BODY_OF[4], True)
check(G, "dates from the TOC",
      lambda: (LETTERS[0]["date"], LETTERS[2]["date_text"], [l["letter_no"] for l in LETTERS if not l["date"]]),
      ("2015-01-19", "21st April, 2015", []))
check(G, "unit: caption found with and without its number",
      lambda: [first_match(t, LETTERHEAD_ANYWHERE) for t in ("3. Dated: 21* April, 2015, JS", "Dated: 21* April, 2015, JS")],
      ["caption", "caption"])
check(G, "unit: short '... Jhalani )' is a sign-off, a long line is not",
      lambda: [is_sign_off(t) for t in ("anoj Jhalani )", "Public Health Facilities (DHs/ SDHs/ CHCs that are FRUs)")],
      ["name", ""])

# ---------- 15.9 text fixes ----------
G = "15.9 fixes"
check(G, "ordinal symbols → st/nd/rd/th",
      lambda: [fix_text(t) for t in ("dated 22™ September", "21* April", '22"4 July', "13'\" July", "19% January", "3° February")],
      ["dated 22nd September", "21st April", "22nd July", "13th July", "19th January", "3rd February"])
check(G, "unit: ordinal() incl. 11-13",
      lambda: [ordinal(n) for n in (1, 2, 3, 11, 12, 13, 21, 22)],
      ["1st", "2nd", "3rd", "11th", "12th", "13th", "21st", "22nd"])
check(G, "no OCR ordinals left in bodies",
      lambda: [no for no, b in BODY_OF.items() if TEXT_FIXES["ordinal"][0].search(b)], [])
check(G, "letter 3 body: dates fixed",
      lambda: "dated 22nd September, 2014 and 19th January, 2015" in BODY_OF[3], True)
check(G, "'| hope' → 'I hope' at paragraph start", lambda: fix_text("| hope that"), "I hope that")
check(G, "bullets → '- ' (but not 'e.g.')",
      lambda: fix_text("e Optimize HR\n\n. Adopting a policy\n\ne.g. not a bullet"),
      "- Optimize HR\n\n- Adopting a policy\n\ne.g. not a bullet")

# ---------- 15.H all 174 pages ----------
G = "15.H pages"
PAGES_OUT = [OCR_PAGES.get(p["pdf_page"], p) for p in PAGES]      # OCR record replaces the empty scanned page
check(G, "174 pages, 56 from OCR",
      lambda: (len(PAGES_OUT), sum(p["method"] == "ocr" for p in PAGES_OUT)), (174, 56))
check(G, "text pages passed through unchanged",
      lambda: [p["pdf_page"] for p in PAGES_OUT if p["method"] != "ocr" and p != PAGE_BY_NO[p["pdf_page"]]], [])

report = pd.DataFrame(TESTS)
n_pass = int((report["result"] == "PASS").sum())
print(f"{'✅' if n_pass == len(report) else '❌'}  {n_pass}/{len(report)} tests passed")
failed = report[report["result"] == "FAIL"]
failed if len(failed) else "All tests passed — nothing to show."

✅  33/33 tests passed


'All tests passed — nothing to show.'

## 15.H Save — only when all tests pass

In [12]:
def write_jsonl(path, records):
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

if n_pass == len(report):
    for path, records in [(PAGES_OCR, PAGES_OUT), (LETTERS_JSONL, LETTERS)]:
        write_jsonl(path, records)
        back = [json.loads(line) for line in path.open(encoding="utf-8")]
        assert back == records, f"{path.name}: what was reloaded differs from what was saved"
        print(f"saved {len(records):>3} records → {path.name} (reload identical)")
else:
    print("NOT saved — fix the failing tests first")

saved 174 records → pages_ocr.jsonl (reload identical)
saved  21 records → letters.jsonl (reload identical)
